# 02 - Feature Engineering
Build 3 analytical datasets:
1. **Product-Week panel** (Q1 sales forecast)
2. **Color-Month panel** (Q2 color forecast)
3. **Customer feature table** (Q3 dealer forecast)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

DATA = Path('../data/processed')

fs = pd.read_parquet(DATA / 'fact_sales_clean.parquet')
products = pd.read_parquet(DATA / 'products.parquet')
prices = pd.read_parquet(DATA / 'product_prices.parquet')

fs['order_date'] = pd.to_datetime(fs['order_date'])

print(f'fact_sales: {fs.shape}')
print(f'Date range: {fs["order_date"].min().date()} -> {fs["order_date"].max().date()}')

fact_sales: (25660, 25)
Date range: 2025-01-02 -> 2026-03-31


## 2.1 Product-Week Panel

In [2]:
fs['week_start'] = fs['order_date'].dt.to_period('W').apply(lambda x: x.start_time)

pw = fs.groupby(['product_code', 'week_start']).agg(
    quantity=('quantity', 'sum'),
    revenue=('line_total', 'sum'),
    unit_price=('unit_price', 'mean'),
    group_code=('group_code', 'first'),
    line_id_fk=('line_id_fk', 'first'),
    line_name=('line_name', 'first'),
    color_std=('color_std', 'first'),
    product_name=('product_name', 'first')
).reset_index()

pw = pw.sort_values(['product_code', 'week_start']).reset_index(drop=True)
print(f'Product-week raw: {pw.shape}')

Product-week raw: (2479, 10)


In [3]:
# Only weeks with at least one sale in the dataset (Apr–Dec 2025 has no records — not zero revenue)
all_weeks = pd.DatetimeIndex(sorted(pw['week_start'].unique()))
all_products = pw['product_code'].unique()

prod_attrs = pw.groupby('product_code').agg(
    group_code=('group_code','first'),
    line_id_fk=('line_id_fk','first'),
    line_name=('line_name','first'),
    color_std=('color_std','first'),
    product_name=('product_name','first'),
    unit_price=('unit_price','last')
).reset_index()

idx = pd.MultiIndex.from_product([all_products, all_weeks], names=['product_code','week_start'])
pw_full = pd.DataFrame(index=idx).reset_index()
pw_full = pw_full.merge(pw[['product_code','week_start','quantity','revenue']], on=['product_code','week_start'], how='left')
# Within an observed week, missing product = no sales for that SKU (0 is correct)
pw_full['quantity'] = pw_full['quantity'].fillna(0)
pw_full['revenue'] = pw_full['revenue'].fillna(0)
pw_full = pw_full.merge(prod_attrs, on='product_code', how='left')

print(f'Observed weeks: {len(all_weeks)} (no gap-filled zero weeks)')
print(f'Full panel: {pw_full.shape}')

Full panel: (16236, 10)


In [4]:
pw_full['month'] = pw_full['week_start'].dt.month
pw_full['week_of_year'] = pw_full['week_start'].dt.isocalendar().week.astype(int)
pw_full['quarter'] = pw_full['week_start'].dt.quarter

pw_full = pw_full.sort_values(['product_code','week_start']).reset_index(drop=True)

for lag in [1, 2, 4]:
    pw_full[f'qty_lag_{lag}w'] = pw_full.groupby('product_code')['quantity'].shift(lag)
    pw_full[f'rev_lag_{lag}w'] = pw_full.groupby('product_code')['revenue'].shift(lag)

for win in [4, 8]:
    pw_full[f'qty_roll_{win}w'] = pw_full.groupby('product_code')['quantity'].transform(
        lambda x: x.rolling(win, min_periods=1).mean())
    pw_full[f'rev_roll_{win}w'] = pw_full.groupby('product_code')['revenue'].transform(
        lambda x: x.rolling(win, min_periods=1).mean())

first_sale = fs.groupby('product_code')['order_date'].min().reset_index()
first_sale.columns = ['product_code', 'first_sale_date']
pw_full = pw_full.merge(first_sale, on='product_code', how='left')
pw_full['weeks_since_first'] = ((pw_full['week_start'] - pw_full['first_sale_date']).dt.days / 7).clip(lower=0)

cum_qty = pw_full.groupby('product_code')['quantity'].cumsum()
pw_full['cum_quantity'] = cum_qty

price_changes = prices.sort_values(['product_code','effective_from'])
price_changes['effective_from'] = pd.to_datetime(price_changes['effective_from'])
pc_count = price_changes.groupby('product_code').size().reset_index(name='n_price_changes')
pw_full = pw_full.merge(pc_count, on='product_code', how='left')
pw_full['n_price_changes'] = pw_full['n_price_changes'].fillna(0)

le_group = LabelEncoder()
pw_full['group_code_enc'] = le_group.fit_transform(pw_full['group_code'].fillna('UNK').astype(str))

le_color = LabelEncoder()
pw_full['color_enc'] = le_color.fit_transform(pw_full['color_std'].fillna('unknown').astype(str))

pw_full = pw_full.drop(columns=['first_sale_date'])
print(f'Product-week panel final: {pw_full.shape}')
print(pw_full.dtypes)

Product-week panel final: (16236, 28)
product_code                    str
week_start           datetime64[us]
quantity                    float64
revenue                     float64
group_code                      str
line_id_fk                  float64
line_name                       str
color_std                       str
product_name                    str
unit_price                  float64
month                         int32
week_of_year                  int64
quarter                       int32
qty_lag_1w                  float64
rev_lag_1w                  float64
qty_lag_2w                  float64
rev_lag_2w                  float64
qty_lag_4w                  float64
rev_lag_4w                  float64
qty_roll_4w                 float64
rev_roll_4w                 float64
qty_roll_8w                 float64
rev_roll_8w                 float64
weeks_since_first           float64
cum_quantity                float64
n_price_changes             float64
group_code_enc            

In [5]:
pw_full.to_parquet(DATA / 'product_week_panel.parquet', index=False)
print('Saved product_week_panel.parquet')

Saved product_week_panel.parquet


## 2.2 Color-Month Panel

In [6]:
fs['ym'] = fs['order_date'].dt.to_period('M')

cm = fs.groupby(['color_std', 'group_code', 'ym']).agg(
    quantity=('quantity','sum'),
    revenue=('line_total','sum')
).reset_index()

totals = fs.groupby(['group_code','ym']).agg(
    total_qty=('quantity','sum'),
    total_rev=('line_total','sum')
).reset_index()

cm = cm.merge(totals, on=['group_code','ym'], how='left')
cm['qty_share'] = cm['quantity'] / cm['total_qty']
cm['rev_share'] = cm['revenue'] / cm['total_rev']

cm['ym_str'] = cm['ym'].astype(str)
cm['month_num'] = cm['ym'].apply(lambda x: x.month)
cm['year'] = cm['ym'].apply(lambda x: x.year)
cm['month_idx'] = (cm['year'] - cm['year'].min()) * 12 + cm['month_num']

cm = cm.sort_values(['color_std','group_code','ym']).reset_index(drop=True)
cm['qty_share_delta'] = cm.groupby(['color_std','group_code'])['qty_share'].diff()

def trend_slope(series):
    s = series.dropna()
    if len(s) < 3:
        return 0.0
    x = np.arange(len(s))
    return np.polyfit(x, s.values, 1)[0]

trend_df = cm.groupby(['color_std','group_code']).agg(
    qty_share_trend=('qty_share', trend_slope),
    rev_share_trend=('rev_share', trend_slope)
).reset_index()

cm = cm.merge(trend_df, on=['color_std','group_code'], how='left')

cm = cm.drop(columns=['ym'])
print(f'Color-month panel: {cm.shape}')
cm.to_parquet(DATA / 'color_month_panel.parquet', index=False)
print('Saved color_month_panel.parquet')

Color-month panel: (344, 15)
Saved color_month_panel.parquet


## 2.3 Customer Feature Table

In [7]:
max_date = fs['order_date'].max()
cutoff = max_date - pd.Timedelta(days=30)
print(f'Max date: {max_date.date()}, Cutoff T-30: {cutoff.date()}')

fs_feat = fs[fs['order_date'] <= cutoff].copy()
fs_label = fs[(fs['order_date'] > cutoff) & (fs['order_date'] <= max_date)].copy()

labels = fs_label.groupby('customer_code')['so_number'].nunique().reset_index()
labels.columns = ['customer_code', 'orders_in_window']
labels['ordered_30d'] = 1

all_custs = fs['customer_code'].unique()
label_df = pd.DataFrame({'customer_code': all_custs})
label_df = label_df.merge(labels[['customer_code','ordered_30d']], on='customer_code', how='left')
label_df['ordered_30d'] = label_df['ordered_30d'].fillna(0).astype(int)
print(f'Label distribution:\n{label_df["ordered_30d"].value_counts()}')

Max date: 2026-03-31, Cutoff T-30: 2026-03-01
Label distribution:
ordered_30d
0    403
1    384
Name: count, dtype: int64


In [8]:
cust_agg = fs_feat.groupby('customer_code').agg(
    last_order=('order_date','max'),
    first_order=('order_date','min'),
    total_orders=('so_number','nunique'),
    total_revenue=('line_total','sum'),
    total_quantity=('quantity','sum'),
    n_skus=('product_code','nunique'),
    n_groups=('group_code','nunique'),
    n_lines=('line_name','nunique'),
    avg_unit_price=('unit_price','mean'),
).reset_index()

cust_agg['recency'] = (cutoff - cust_agg['last_order']).dt.days
cust_agg['tenure'] = (cutoff - cust_agg['first_order']).dt.days
cust_agg['avg_order_value'] = cust_agg['total_revenue'] / cust_agg['total_orders']
cust_agg['avg_qty_per_order'] = cust_agg['total_quantity'] / cust_agg['total_orders']

active_months = (cust_agg['tenure'] / 30).clip(lower=1)
cust_agg['orders_per_month'] = cust_agg['total_orders'] / active_months
cust_agg['revenue_per_month'] = cust_agg['total_revenue'] / active_months

In [9]:
mid = cutoff - pd.Timedelta(days=90)

recent = fs_feat[fs_feat['order_date'] > mid].groupby('customer_code').agg(
    orders_recent=('so_number','nunique'),
    rev_recent=('line_total','sum')
).reset_index()

prior = fs_feat[fs_feat['order_date'] <= mid].groupby('customer_code').agg(
    orders_prior=('so_number','nunique'),
    rev_prior=('line_total','sum')
).reset_index()

cust_agg = cust_agg.merge(recent, on='customer_code', how='left')
cust_agg = cust_agg.merge(prior, on='customer_code', how='left')

for c in ['orders_recent','rev_recent','orders_prior','rev_prior']:
    cust_agg[c] = cust_agg[c].fillna(0)

cust_agg['order_trend'] = cust_agg['orders_recent'] - cust_agg['orders_prior']
cust_agg['revenue_trend'] = cust_agg['rev_recent'] - cust_agg['rev_prior']

In [10]:
def monthly_slope(group):
    monthly = group.set_index('order_date').resample('ME')['line_total'].sum()
    if len(monthly) < 3:
        return pd.Series({'rev_slope': 0.0, 'qty_slope': 0.0})
    x = np.arange(len(monthly))
    rev_slope = np.polyfit(x, monthly.values, 1)[0]
    monthly_q = group.set_index('order_date').resample('ME')['quantity'].sum()
    qty_slope = np.polyfit(x, monthly_q.values, 1)[0]
    return pd.Series({'rev_slope': rev_slope, 'qty_slope': qty_slope})

slopes = fs_feat.groupby('customer_code').apply(monthly_slope, include_groups=False).reset_index()
cust_agg = cust_agg.merge(slopes, on='customer_code', how='left')
cust_agg['rev_slope'] = cust_agg['rev_slope'].fillna(0)
cust_agg['qty_slope'] = cust_agg['qty_slope'].fillna(0)

In [11]:
region_map = fs_feat.groupby('customer_code')['region'].first().reset_index()
cust_agg = cust_agg.merge(region_map, on='customer_code', how='left')

le_region = LabelEncoder()
cust_agg['region_enc'] = le_region.fit_transform(cust_agg['region'].fillna('Unknown').astype(str))

cust_agg = cust_agg.merge(label_df, on='customer_code', how='left')
cust_agg['ordered_30d'] = cust_agg['ordered_30d'].fillna(0).astype(int)

cust_agg = cust_agg.drop(columns=['last_order','first_order'])

print(f'Customer features: {cust_agg.shape}')
print(cust_agg.columns.tolist())
cust_agg.to_parquet(DATA / 'customer_features.parquet', index=False)
print('Saved customer_features.parquet')

Customer features: (678, 25)
['customer_code', 'total_orders', 'total_revenue', 'total_quantity', 'n_skus', 'n_groups', 'n_lines', 'avg_unit_price', 'recency', 'tenure', 'avg_order_value', 'avg_qty_per_order', 'orders_per_month', 'revenue_per_month', 'orders_recent', 'rev_recent', 'orders_prior', 'rev_prior', 'order_trend', 'revenue_trend', 'rev_slope', 'qty_slope', 'region', 'region_enc', 'ordered_30d']
Saved customer_features.parquet


In [12]:
print('=== All feature datasets ===')
for f in sorted(DATA.glob('*.parquet')):
    df = pd.read_parquet(f)
    print(f'{f.name}: {df.shape}')

=== All feature datasets ===
color_month_panel.parquet: (344, 15)
customer_features.parquet: (678, 25)
customers.parquet: (798, 7)
fact_sales_clean.parquet: (25660, 25)
product_groups.parquet: (5, 3)
product_lines.parquet: (77, 3)
product_prices.parquet: (1016, 4)
product_week_panel.parquet: (16236, 28)
products.parquet: (247, 9)
provinces.parquet: (75, 3)
sales_orders.parquet: (2759, 12)
